In [ ]:
import sys, os
while not os.path.isdir('src') and os.path.dirname(os.getcwd()) != os.getcwd():
    os.chdir('..')
sys.path.insert(0, 'src')

import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

from tradinglab.data_feed import DataFeed
from tradinglab.features import build_pooled_dataset
from tradinglab.models import MLP
from tradinglab.simulator import PortfolioSimulator
from tradinglab.backtester import run_backtest
from tradinglab.metrics import total_return, sharpe, max_drawdown

# MLP Task 2 — Trade every 10 days

Train MLP on pooled 34-stock data, then trade with 10-day rebalance period.

In [ ]:
# Load data
feed = DataFeed.from_dir('data/egx')
print(f"Universe: {feed.n_assets} assets")
print(f"Period: {feed.dates[0].date()} → {feed.dates[-1].date()}")
print(f"Days: {feed.n_days}")

In [ ]:
# Compute split day (70% of calendar days)
split_day = int(feed.n_days * 0.7)

# Build pooled dataset
X_train, y_train, X_test, y_test = build_pooled_dataset(feed, split_day)
print(f"X_train shape: {X_train.shape}  (samples, features)")
print(f"y_train shape: {y_train.shape}")
print(f"X_test shape:  {X_test.shape}")
print(f"y_test shape:  {y_test.shape}")

print(f"\nTrain: {X_train.shape[0]} samples")
print(f"Test:  {X_test.shape[0]} samples")

In [ ]:
# Train MLP
n_features = X_train.shape[1]
model = MLP(n_features, hidden=32)

X_tr = torch.tensor(X_train, dtype=torch.float32)
y_tr = torch.tensor(y_train, dtype=torch.float32)
X_te = torch.tensor(X_test, dtype=torch.float32)
y_te = torch.tensor(y_test, dtype=torch.float32)

opt = torch.optim.Adam(model.parameters(), lr=0.001)
loss_fn = nn.MSELoss()

epochs = 1500
for epoch in range(epochs):
    opt.zero_grad()
    pred = model(X_tr)
    loss = loss_fn(pred, y_tr)
    loss.backward()
    opt.step()
    
    if (epoch + 1) % 300 == 0:
        with torch.no_grad():
            test_loss = loss_fn(model(X_te), y_te).item()
        print(f"Epoch {epoch+1}/{epochs}  train loss={loss.item():.6f}  test loss={test_loss:.6f}")

print(f"\nFinal train loss: {loss.item():.6f}")
print(f"Final test loss:  {test_loss:.6f}")

## Strategy — Rebalance every 10 days

In [ ]:
class MLPStrategyRebalance10:
    """Buy top-5 stocks from MLP predictions, hold 10 days, then rebalance."""
    
    def __init__(self, model, feed, top_k=5):
        self.model = model
        self.feed = feed
        self.top_k = top_k
        self.day_count = 0
        self.current_weights = None
    
    def __call__(self, observation):
        """observation shape: (34, 9) — 34 stocks, 9 features each."""
        self.day_count += 1
        
        # Rebalance every 10 days
        if self.day_count % 10 == 1:
            n_assets = observation.shape[0]
            
            with torch.no_grad():
                X_torch = torch.as_tensor(observation, dtype=torch.float32)
                predictions = self.model(X_torch).numpy()  # shape: (34,)
            
            # Buy top-5 predicted returns
            top_indices = np.argsort(predictions)[-self.top_k:]
            weights = np.zeros(n_assets)
            weights[top_indices] = 1.0 / self.top_k  # Equal weight to top-5
            self.current_weights = weights
        
        return self.current_weights if self.current_weights is not None else np.zeros(34)

In [ ]:
# Run backtest with commission
COMMISSION = 0.005  # 0.5%

strategy = MLPStrategyRebalance10(model, feed, top_k=5)
sim = PortfolioSimulator(feed, benchmark='egx30', commission=COMMISSION)
result_mlp = run_backtest(sim, strategy, lookback=30, start=split_day)

portfolio_val = result_mlp['portfolio'] * 1000
equity = np.array(result_mlp['portfolio'])
rets = np.diff(equity) / equity[:-1]

tr = total_return(rets)
sh = sharpe(rets)
md = max_drawdown(rets)

print(f"MLP Rebalance 10-day:")
print(f"Final value:  {portfolio_val[-1]:>10,.0f} EGP")
print(f"Total return: {tr:>10.1%}")
print(f"Sharpe:       {sh:>10.2f}")
print(f"Max drawdown: {md:>10.1%}")

In [ ]:
# Plot equity curve
plt.figure(figsize=(13, 6))

START = 1000.0
plt.plot(result_mlp['dates'], result_mlp['benchmark']*START, 
         label='EGX30 (benchmark)', linestyle='--', linewidth=2, color='gray')
plt.plot(result_mlp['dates'], portfolio_val, 
         label=f'MLP 10-day Rebalance ({tr:.0%})', linewidth=2, color='#ff7f0e')

plt.xlabel('Date')
plt.ylabel('Portfolio Value (EGP)')
plt.title('MLP Strategy — 10-Day Rebalance with 0.5% Commission')
plt.legend(loc='best')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()